# Fase 4.1 — Modelo dimensional para Power BI

Los informes recibidos confirman que `RandomForest_log` mejora el MAE de prueba frente a la mediana en 21,71 %, con MAE BRL 594,06, error mediano BRL 128,20 y R² 0,01514. La evaluación hasta el P99 de desarrollo tiene otra población (9 098 casos) y no sustituye los 9 206 de prueba completa.

Este notebook integra la oferta con sus estimaciones y crea un modelo dimensional para un **estudio de mercado con estimaciones experimentales**. No reentrena el modelo ni cambia su evaluación.

## Ejecución y escritura

Guarda el archivo en `notebooks/` y ejecuta las **10 celdas** con tu `.venv`. Requiere los archivos locales ya creados en `reports/modelado/`, especialmente `estimaciones_para_powerbi.csv`. No hace falta subir aquí ese archivo completo.

La **celda 8 crea 10 tablas nuevas en el esquema `bi`** de PostgreSQL, con claves primarias y relaciones. Falla si cualquiera de esas tablas ya existe y no borra ni añade filas a tablas previas. Conserva `staging` y `clean`. Las celdas previas preparan y validan los datos en memoria.

La celda 9 genera `powerbi/GUIA_POWERBI.md` y `powerbi/medidas_airbnb.dax`. Este notebook prepara los datos; el archivo `.pbix` se construye después en tu Power BI Desktop siguiendo esa guía.

## 1. Configurar rutas y conexión

In [1]:
import sys
import json
from pathlib import Path
from datetime import datetime, timezone
from getpass import getpass

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.engine import URL
from sqlalchemy.types import Text, BigInteger, Float, Boolean, Date, Numeric
from IPython.display import display

HOST,PORT='localhost',5432
DATABASE,USER='proy_airbnb_rio','postgres'
RAIZ_MANUAL=''
if RAIZ_MANUAL:RAIZ=Path(RAIZ_MANUAL).expanduser().resolve()
else:
    inicio=Path.cwd().resolve()
    RAIZ=next((p for p in [inicio,*inicio.parents] if (p/'data'/'raw').is_dir() and (p/'src').is_dir()),None)
if RAIZ is None:raise RuntimeError('Configura RAIZ_MANUAL con la carpeta principal del proyecto.')
MODELADO=RAIZ/'reports'/'modelado'
REPORTES=RAIZ/'reports'/'powerbi'
POWERBI=RAIZ/'powerbi'
for p in [REPORTES,POWERBI]:p.mkdir(parents=True,exist_ok=True)
print('Python:',sys.executable)
print('Archivos del modelo:',MODELADO)
print('Destino PostgreSQL:',DATABASE,'→ bi')

Python: c:\Users\user\Documents\Inteligencia de negocios\Proyecto_IN\Proy_Analysis_Airbnb_Business_Model\.venv\Scripts\python.exe
Archivos del modelo: C:\Users\user\Documents\Inteligencia de negocios\Proyecto_IN\Proy_Analysis_Airbnb_Business_Model\reports\modelado
Destino PostgreSQL: proy_airbnb_rio → bi


## 2. Leer la oferta y las salidas locales del modelo

La lectura de PostgreSQL es de solo lectura. El modelo debe corresponder a la misma oferta y corte. Se conservarán los identificadores como texto.

In [2]:
archivos=['estimaciones_para_powerbi.csv','resumen_modelado.json','comparacion_validacion.csv',
          'metricas_prueba.csv','sensibilidad_prueba.csv','importancia_variables.csv']
faltan=[n for n in archivos if not (MODELADO/n).is_file()]
if faltan:raise FileNotFoundError(f'Faltan archivos en reports/modelado: {faltan}')
resumen_modelo=json.loads((MODELADO/'resumen_modelado.json').read_text(encoding='utf-8-sig'))
estimaciones=pd.read_csv(MODELADO/'estimaciones_para_powerbi.csv',dtype={'id':'string'})
validacion=pd.read_csv(MODELADO/'comparacion_validacion.csv')
prueba=pd.read_csv(MODELADO/'metricas_prueba.csv')
sensibilidad=pd.read_csv(MODELADO/'sensibilidad_prueba.csv')
importancias=pd.read_csv(MODELADO/'importancia_variables.csv')
motor=create_engine(URL.create('postgresql+psycopg2',username=USER,
    password=getpass(f'Contraseña de {USER}: '),host=HOST,port=PORT,database=DATABASE),
    connect_args={'connect_timeout':10})
try:
    with motor.connect() as conexion:
        with conexion.begin():
            conexion.execute(text('SET TRANSACTION READ ONLY'))
            oferta=pd.read_sql_query(text('SELECT * FROM clean.listings'),conexion)
except Exception:
    motor.dispose();raise
print('Oferta leída:',len(oferta),'Estimaciones:',len(estimaciones))

Oferta leída: 48247 Estimaciones: 48247


## 3. Integrar por ID y comprobar métricas de prueba

Se exige correspondencia uno-a-uno y coincidencia del precio observado entre el CSV y la tabla. Se reconstruye la referencia de mediana con desarrollo, según el protocolo del modelo; no es un entrenamiento nuevo. Los errores se guardarán únicamente para filas de prueba.

In [3]:
requeridas={'id','host_id','scrape_id','last_scraped','neighbourhood_cleansed','room_type','property_type',
            'accommodates','price','bedrooms','beds','bathrooms_derived','latitude','longitude',
            'minimum_nights','maximum_nights','number_of_reviews','review_scores_rating',
            'availability_30','availability_365','capacity_needs_review','nights_needs_review'}
if not requeridas<=set(oferta):raise ValueError(f'Faltan columnas de clean: {sorted(requeridas-set(oferta))}')
cols_pred=['id','precio_observado_BRL','precio_estimado_BRL','particion_original',
           'comparables_desarrollo','soporte_bajo_segmento','modelo']
if not set(cols_pred)<=set(estimaciones):raise ValueError('El CSV de estimaciones no tiene la estructura esperada.')
oferta=oferta.copy().reset_index(drop=True)
oferta['id']=oferta['id'].astype('string')
if oferta['id'].isna().any() or oferta['id'].duplicated().any() or estimaciones['id'].isna().any() or estimaciones['id'].duplicated().any():
    raise ValueError('Hay IDs ausentes o repetidos.')
if set(oferta['id'])!=set(estimaciones['id']):raise ValueError('Los IDs de oferta y estimaciones no coinciden.')
if oferta['scrape_id'].isna().any() or oferta['scrape_id'].nunique()!=1:
    raise ValueError('Esta preparación está diseñada para un solo corte identificado.')
integrado=oferta.merge(estimaciones[cols_pred],on='id',how='left',validate='one_to_one')
precio=pd.to_numeric(integrado['price'],errors='coerce').astype(float)
precio_csv=pd.to_numeric(integrado['precio_observado_BRL'],errors='coerce').astype(float)
if not np.allclose(precio,precio_csv,rtol=0,atol=.0001,equal_nan=True):
    raise ValueError('El precio observado del modelo difiere de clean.listings.')
estimado=pd.to_numeric(integrado['precio_estimado_BRL'],errors='coerce').astype(float)
if not (np.isfinite(estimado)&estimado.gt(0)).all():raise ValueError('Hay estimaciones no válidas.')
permitidas={'entrenamiento','validacion','prueba','sin_precio'}
if not set(integrado['particion_original'])<=permitidas:raise ValueError('Particiones desconocidas.')
for p in resumen_modelo['particiones']:
    if int(integrado['particion_original'].eq(p['particion']).sum())!=p['anuncios']:
        raise ValueError('Los conteos de particiones difieren del resumen del modelo.')
if not integrado['particion_original'].eq('sin_precio').equals(precio.isna()):
    raise ValueError('Los anuncios sin precio no coinciden con la partición declarada.')
if len(integrado)!=resumen_modelo['anuncios_totales'] or not integrado['modelo'].eq(resumen_modelo['modelo_elegido']).all():
    raise ValueError('La oferta o versión del modelo no coincide con su resumen.')
es_prueba=integrado['particion_original'].eq('prueba')
es_desarrollo=integrado['particion_original'].isin(['entrenamiento','validacion'])
grupos_host=integrado['host_id'].astype('string').str.strip()
grupos_host=grupos_host.mask(grupos_host.isna()|grupos_host.eq('').fillna(False),'<HOST_AUSENTE>_'+integrado['id'])
grupos_particion={p:set(grupos_host.loc[integrado['particion_original'].eq(p)]) for p in ['entrenamiento','validacion','prueba']}
for a,b in [('entrenamiento','validacion'),('entrenamiento','prueba'),('validacion','prueba')]:
    if grupos_particion[a]&grupos_particion[b]:raise ValueError('Un anfitrión aparece en dos particiones del modelo.')
referencia=float(precio.loc[es_desarrollo].median())
errores=(precio-estimado).abs()
error_referencia=(precio-referencia).abs()
mae_calculado=float(errores.loc[es_prueba].mean())
rmse_calculado=float(np.sqrt(((precio-estimado).loc[es_prueba]**2).mean()))
mae_ref_calculado=float(error_referencia.loc[es_prueba].mean())
for actual,esperado in [(mae_calculado,resumen_modelo['metricas_prueba_modelo']['MAE_BRL']),
                       (rmse_calculado,resumen_modelo['metricas_prueba_modelo']['RMSE_BRL']),
                       (mae_ref_calculado,resumen_modelo['metricas_prueba_referencia']['MAE_BRL'])]:
    if not np.isclose(actual,esperado,rtol=1e-7,atol=1e-5):raise ValueError('Las métricas reconstruidas no coinciden con el modelo.')
print('Integración uno-a-uno verificada. MAE prueba:',mae_calculado)
print('Mediana de desarrollo usada como referencia:',referencia)

Integración uno-a-uno verificada. MAE prueba: 594.0603346122907
Mediana de desarrollo usada como referencia: 450.0


## 4. Construir las siete dimensiones

Las claves numéricas se generan para este corte. Esto no implementa una carga incremental ni cambios históricos de dimensiones. La fecha corresponde a recopilación, no a una fecha de reserva.

In [4]:
def dimension(serie,columna,clave):
    valores=serie.astype('string').str.strip().replace('',pd.NA).fillna('<SIN DATO>')
    d=pd.DataFrame({columna:sorted(valores.unique())})
    d.insert(0,clave,np.arange(1,len(d)+1,dtype='int64'))
    return d,valores.map(d.set_index(columna)[clave]).astype('int64')

dim_barrio,k_barrio=dimension(integrado['neighbourhood_cleansed'],'barrio','barrio_key')
dim_tipo,k_tipo=dimension(integrado['room_type'],'tipo_alojamiento','tipo_alojamiento_key')
dim_tipo['tipo_alojamiento_es']=dim_tipo['tipo_alojamiento'].map({
    'Entire home/apt':'Alojamiento completo','Private room':'Habitación privada',
    'Shared room':'Habitación compartida','Hotel room':'Habitación de hotel'}).fillna(dim_tipo['tipo_alojamiento'])
dim_propiedad,k_propiedad=dimension(integrado['property_type'],'tipo_propiedad','tipo_propiedad_key')
grupo_capacidad=pd.cut(pd.to_numeric(integrado['accommodates'],errors='coerce'),
    bins=[0,2,4,6,8,np.inf],labels=['1-2','3-4','5-6','7-8','9+']).astype('string').fillna('<SIN DATO>')
dim_capacidad,k_capacidad=dimension(grupo_capacidad,'grupo_capacidad','capacidad_key')
dim_capacidad['orden']=dim_capacidad['grupo_capacidad'].map({'1-2':1,'3-4':2,'5-6':3,'7-8':4,'9+':5}).fillna(99).astype('int64')
dim_anfitrion,k_anfitrion=dimension(integrado['host_id'],'host_id','anfitrion_key')
fechas=pd.to_datetime(integrado['last_scraped'],errors='coerce')
if fechas.isna().any():raise ValueError('Hay fechas de recopilación no válidas.')
calendario=pd.Series(pd.date_range(f'{fechas.min().year}-01-01',f'{fechas.max().year}-12-31',freq='D'))
dim_fecha=pd.DataFrame({'fecha_key':calendario.dt.strftime('%Y%m%d').astype('int64'),
    'fecha':calendario.dt.date,'anio':calendario.dt.year.astype('int64'),
    'mes':calendario.dt.month.astype('int64'),'dia':calendario.dt.day.astype('int64'),
    'dia_semana':(calendario.dt.dayofweek+1).astype('int64')})
meses=['Enero','Febrero','Marzo','Abril','Mayo','Junio','Julio','Agosto','Septiembre','Octubre','Noviembre','Diciembre']
dim_fecha['mes_nombre']=dim_fecha['mes'].map(dict(enumerate(meses,start=1)))
corte_id=str(integrado['scrape_id'].iloc[0])
dim_corte=pd.DataFrame([{'corte_id':corte_id,'recopilacion_desde':fechas.min().date(),
    'recopilacion_hasta':fechas.max().date(),'fuente':'Inside Airbnb','moneda':'BRL'}])
print('Dimensiones preparadas:',len(dim_barrio),'barrios,',len(dim_anfitrion),'identificadores de anfitrión.')

Dimensiones preparadas: 156 barrios, 26869 identificadores de anfitrión.


C:\Users\user\AppData\Local\Temp\ipykernel_4576\1212312202.py:24: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  'dia_semana':(calendario.dt.dayofweek+1).astype('int64')})


## 5. Construir la tabla de hechos sin duplicar anuncios

Los errores de desarrollo y de anuncios sin precio quedan NULL. Las diferencias publicadas frente al modelo se conservan para descripción, separadas del error de prueba. No se etiquetan como sobreprecio o infravaloración.

In [5]:
def booleano(serie):
    s=serie.astype('string').str.lower().str.strip()
    valores={'true':True,'t':True,'1':True,'false':False,'f':False,'0':False}
    if (s.notna()&~s.isin(valores)).any():raise ValueError('Booleano no reconocido.')
    return s.map(valores).astype('boolean')

fact=pd.DataFrame({'anuncio_id':integrado['id'].astype(str),'corte_id':corte_id,
    'barrio_key':k_barrio,'tipo_alojamiento_key':k_tipo,'tipo_propiedad_key':k_propiedad,
    'capacidad_key':k_capacidad,'anfitrion_key':k_anfitrion,
    'fecha_key':fechas.dt.strftime('%Y%m%d').astype('int64'),
    'precio_publicado':precio,'precio_estimado':estimado,
    'sin_precio':precio.isna(),'particion_modelo':integrado['particion_original'],
    'incluido_ajuste_final':es_desarrollo,'modelo':integrado['modelo'],
    'diferencia_publicado_estimado':precio-estimado,
    'desviacion_relativa_modelo':(precio-estimado)/estimado,
    'comparables_desarrollo':pd.to_numeric(integrado['comparables_desarrollo'],errors='raise').astype('int64'),
    'soporte_bajo_segmento':booleano(integrado['soporte_bajo_segmento']),
    'error_absoluto_prueba':errores.where(es_prueba),
    'error_cuadratico_prueba':((precio-estimado)**2).where(es_prueba),
    'error_referencia_prueba':error_referencia.where(es_prueba),
    'prueba_sensibilidad':es_prueba&precio.le(resumen_modelo['umbral_P99_desarrollo_BRL']),
    'alerta_capacidad':booleano(integrado['capacity_needs_review']),
    'alerta_noches':booleano(integrado['nights_needs_review']),
})
columnas_numericas={'capacidad':'accommodates','dormitorios':'bedrooms','camas':'beds',
    'banos':'bathrooms_derived','latitud':'latitude','longitud':'longitude',
    'noches_minimas':'minimum_nights','noches_maximas':'maximum_nights',
    'resenas':'number_of_reviews','valoracion':'review_scores_rating',
    'disponibilidad_30':'availability_30','disponibilidad_365':'availability_365'}
enteras={'capacidad','dormitorios','camas','noches_minimas','noches_maximas','resenas','disponibilidad_30','disponibilidad_365'}
for destino,origen in columnas_numericas.items():
    valores=pd.to_numeric(integrado[origen],errors='coerce')
    fact[destino]=valores.astype('Int64' if destino in enteras else 'float64')
if len(fact)!=len(oferta) or fact.duplicated(['corte_id','anuncio_id']).any():raise ValueError('Grano de hechos no válido.')
if fact.loc[~es_prueba,'error_absoluto_prueba'].notna().any():raise ValueError('Se mezclaron errores de desarrollo con prueba.')
if int(fact['prueba_sensibilidad'].sum())!=resumen_modelo['filas_prueba_sensibilidad']:
    raise ValueError('La población de sensibilidad no coincide con su evaluación.')
if not fact['soporte_bajo_segmento'].equals(fact['comparables_desarrollo'].lt(30).astype('boolean')):
    raise ValueError('El indicador de soporte no coincide con su regla.')
display(fact[['anuncio_id','precio_publicado','precio_estimado','particion_modelo']].head())

,anuncio_id,precio_publicado,precio_estimado,particion_modelo
0,17878,513.00,624.745924,entrenamiento
1,25026,422.00,418.050842,entrenamiento
2,35764,366.00,440.706124,entrenamiento
3,48305,3761.43,6078.063764,entrenamiento
4,48901,1173.25,1657.979841,entrenamiento


## 6. Preparar tablas auxiliares y tipos SQL

Las métricas globales y la importancia de variables se almacenan aparte. En Power BI no se relacionan con la tabla de hechos; las medidas basadas en el hecho permiten evaluar subconjuntos seleccionados de prueba.

In [6]:
conteos={x['particion']:x['anuncios'] for x in resumen_modelo['particiones']}
aux=[]
for fase,poblacion,datos_aux,n in [
    ('validacion','Validación',validacion,conteos['validacion']),
    ('prueba','Prueba completa',prueba,conteos['prueba']),
    ('sensibilidad','Hasta P99 de desarrollo; sensibilidad',sensibilidad,resumen_modelo['filas_prueba_sensibilidad'])]:
    t=datos_aux[['modelo','MAE_BRL','RMSE_BRL','MedAE_BRL','R2']].rename(columns={
        'MAE_BRL':'mae_brl','RMSE_BRL':'rmse_brl','MedAE_BRL':'medae_brl','R2':'r2'}).copy()
    t['fase']=fase;t['poblacion']=poblacion;t['anuncios']=int(n);aux.append(t)
metricas_modelo=pd.concat(aux,ignore_index=True)
metricas_modelo.insert(0,'metrica_id',np.arange(1,len(metricas_modelo)+1,dtype='int64'))
importancia_variables=importancias.rename(columns={'aumento_MAE_BRL':'aumento_mae_brl'}).copy()
importancia_variables['modelo']=resumen_modelo['modelo_elegido']
importancia_variables['poblacion']='Muestra de validación usada para selección'
tablas={'dim_barrio':dim_barrio,'dim_tipo_alojamiento':dim_tipo,'dim_tipo_propiedad':dim_propiedad,
    'dim_capacidad':dim_capacidad,'dim_anfitrion':dim_anfitrion,'dim_fecha':dim_fecha,'dim_corte':dim_corte,
    'fact_oferta':fact,'metricas_modelo':metricas_modelo,'importancia_variables':importancia_variables}
claves_primarias={'dim_barrio':['barrio_key'],'dim_tipo_alojamiento':['tipo_alojamiento_key'],
    'dim_tipo_propiedad':['tipo_propiedad_key'],'dim_capacidad':['capacidad_key'],
    'dim_anfitrion':['anfitrion_key'],'dim_fecha':['fecha_key'],'dim_corte':['corte_id'],
    'fact_oferta':['corte_id','anuncio_id'],'metricas_modelo':['metrica_id'],'importancia_variables':['variable']}
relaciones=[('barrio_key','dim_barrio','barrio_key'),('tipo_alojamiento_key','dim_tipo_alojamiento','tipo_alojamiento_key'),
    ('tipo_propiedad_key','dim_tipo_propiedad','tipo_propiedad_key'),('capacidad_key','dim_capacidad','capacidad_key'),
    ('anfitrion_key','dim_anfitrion','anfitrion_key'),('fecha_key','dim_fecha','fecha_key'),('corte_id','dim_corte','corte_id')]
tipos={}
for nombre,t in tablas.items():
    tipos[nombre]={}
    for c in t:
        if c=='precio_publicado':tipo=Numeric(20,2)
        elif pd.api.types.is_bool_dtype(t[c]):tipo=Boolean()
        elif pd.api.types.is_integer_dtype(t[c]):tipo=BigInteger()
        elif pd.api.types.is_float_dtype(t[c]):tipo=Float()
        elif c in ['fecha','recopilacion_desde','recopilacion_hasta']:tipo=Date()
        else:tipo=Text()
        tipos[nombre][c]=tipo
print('Tablas previstas:',list(tablas))

Tablas previstas: ['dim_barrio', 'dim_tipo_alojamiento', 'dim_tipo_propiedad', 'dim_capacidad', 'dim_anfitrion', 'dim_fecha', 'dim_corte', 'fact_oferta', 'metricas_modelo', 'importancia_variables']


## 7. Validar claves, cobertura y consistencia antes de cargar

In [7]:
for nombre,t in tablas.items():
    pk=claves_primarias[nombre]
    if t[pk].isna().any().any() or t.duplicated(pk).any():raise ValueError(f'Clave primaria no válida: {nombre}')
for campo,dim,campo_dim in relaciones:
    if fact[campo].isna().any() or not fact[campo].isin(tablas[dim][campo_dim]).all():
        raise ValueError(f'Hay referencias sin dimensión: {campo}')
catalogo=pd.DataFrame([{'tabla':nombre,'filas':len(t),'columnas':len(t.columns),
    'clave_primaria':', '.join(claves_primarias[nombre])} for nombre,t in tablas.items()])
validaciones=pd.DataFrame([
    {'control':'anuncios','valor':len(fact)},
    {'control':'anuncios_con_precio','valor':int(fact['precio_publicado'].gt(0).sum())},
    {'control':'anuncios_sin_precio','valor':int(fact['sin_precio'].sum())},
    {'control':'anuncios_evaluados_prueba','valor':int(fact['error_absoluto_prueba'].notna().sum())},
    {'control':'MAE_prueba_BRL','valor':float(fact['error_absoluto_prueba'].mean())},
    {'control':'MAE_referencia_prueba_BRL','valor':float(fact['error_referencia_prueba'].mean())},
    {'control':'anuncios_sensibilidad','valor':int(fact['prueba_sensibilidad'].sum())},
])
resumen_powerbi={'generado_utc':datetime.now(timezone.utc).isoformat(),'esquema':'bi',
    'modelo':resumen_modelo['modelo_elegido'],'moneda':'BRL','grano':'Un anuncio por corte',
    'corte_id':corte_id,'tablas':catalogo.to_dict(orient='records'),
    'validaciones':validaciones.to_dict(orient='records'),'estado_carga':'pendiente',
    'guia':'powerbi/GUIA_POWERBI.md','medidas':'powerbi/medidas_airbnb.dax'}
def guardar_informes():
    catalogo.to_csv(REPORTES/'catalogo_tablas_bi.csv',index=False,encoding='utf-8-sig')
    validaciones.to_csv(REPORTES/'validaciones_bi.csv',index=False,encoding='utf-8-sig')
    (REPORTES/'resumen_powerbi.json').write_text(json.dumps(resumen_powerbi,ensure_ascii=False,indent=2),encoding='utf-8')
guardar_informes()
display(catalogo);display(validaciones)

,tabla,filas,columnas,clave_primaria
0,dim_barrio,156,2,barrio_key
1,dim_tipo_alojamiento,4,3,tipo_alojamiento_key
2,dim_tipo_propiedad,80,2,tipo_propiedad_key
3,dim_capacidad,5,3,capacidad_key
4,dim_anfitrion,26869,2,anfitrion_key
5,dim_fecha,365,7,fecha_key
6,dim_corte,1,5,corte_id
7,fact_oferta,48247,36,"corte_id, anuncio_id"
8,metricas_modelo,7,9,metrica_id
9,importancia_variables,12,5,variable


,control,valor
0,anuncios,48247.000000
1,anuncios_con_precio,44147.000000
2,anuncios_sin_precio,4100.000000
3,anuncios_evaluados_prueba,9206.000000
4,MAE_prueba_BRL,594.060335
5,MAE_referencia_prueba_BRL,758.794134
6,anuncios_sensibilidad,9098.000000


## 8. Crear y cargar las tablas en PostgreSQL

**Esta celda escribe en `bi`.** Todas las tablas se crean y cargan dentro de una transacción. Si ya existe cualquiera de las tablas previstas, se detiene sin borrarlas ni agregar registros. Las claves primarias y foráneas impiden duplicaciones y referencias sin dimensión.

In [8]:
try:
    with motor.begin() as conexion:
        existentes=[n for n in tablas if inspect(conexion).has_table(n,schema='bi')]
        if existentes:raise RuntimeError(f'Ya existen tablas de destino: {existentes}. No se sobrescribirán.')
        conexion.execute(text('CREATE SCHEMA IF NOT EXISTS bi'))
        for nombre,t in tablas.items():
            t.to_sql(nombre,con=conexion,schema='bi',if_exists='fail',index=False,
                     dtype=tipos[nombre],chunksize=500,method='multi')
            pk=', '.join(claves_primarias[nombre])
            conexion.execute(text(f'ALTER TABLE bi.{nombre} ADD PRIMARY KEY ({pk})'))
            cargadas=int(conexion.execute(text(f'SELECT COUNT(*) FROM bi.{nombre}')).scalar_one())
            if cargadas!=len(t):raise RuntimeError(f'Conteo diferente en {nombre}; se revierte.')
        for campo,dim,campo_dim in relaciones:
            conexion.execute(text(f'ALTER TABLE bi.fact_oferta ADD FOREIGN KEY ({campo}) REFERENCES bi.{dim} ({campo_dim})'))
    resumen_powerbi['estado_carga']='completada'
    guardar_informes()
    print('Carga confirmada: 10 tablas nuevas en bi, con claves y relaciones.')
except Exception:
    motor.dispose();raise

Carga confirmada: 10 tablas nuevas en bi, con claves y relaciones.


## 9. Generar la guía y las medidas de Power BI

Los archivos generados indican cómo conectar, relacionar, crear medidas y diseñar cuatro páginas. Se reemplazan si vuelves a ejecutar esta celda. No se genera automáticamente un PBIX.

In [9]:
GUIA_POWERBI='# Construcción del informe en Power BI Desktop\n\n## 1. Conectar al modelo dimensional\n\nInicio → Obtener datos → Base de datos PostgreSQL. Servidor: `localhost:5432`. Base: `proy_airbnb_rio`. Seleccionar **Importar** y autenticación de base de datos con el usuario y contraseña de tu PostgreSQL. Mantener las credenciales en el diálogo, no en archivos de código.\n\nEn Navegador selecciona las 10 tablas del esquema `bi`. En Power Query, utiliza estos nombres de consulta, sin prefijo `bi`, para que coincidan con las medidas: `dim_barrio`, `dim_tipo_alojamiento`, `dim_tipo_propiedad`, `dim_capacidad`, `dim_anfitrion`, `dim_fecha`, `dim_corte`, `fact_oferta`, `metricas_modelo`, `importancia_variables`.\n\nComprueba que `anuncio_id`, `host_id` y `corte_id` sean **Texto**. Son identificadores, no cantidades. Los precios deben ser numéricos y las fechas de tipo Fecha. Cerrar y aplicar.\n\n## 2. Relaciones y granularidad\n\nGrano de `fact_oferta`: un anuncio en un corte. En esta entrega hay un solo corte. Crea o comprueba estas relaciones activas 1:* con filtro en una dirección, dimensión → hecho:\n\n| Dimensión, columna única | Columna de fact_oferta |\n|---|---|\n| dim_barrio[barrio_key] | barrio_key |\n| dim_tipo_alojamiento[tipo_alojamiento_key] | tipo_alojamiento_key |\n| dim_tipo_propiedad[tipo_propiedad_key] | tipo_propiedad_key |\n| dim_capacidad[capacidad_key] | capacidad_key |\n| dim_anfitrion[anfitrion_key] | anfitrion_key |\n| dim_fecha[fecha_key] | fecha_key |\n| dim_corte[corte_id] | corte_id |\n\n`metricas_modelo` e `importancia_variables` quedan **sin relaciones**. Son resultados globales de una ejecución, no observaciones individuales. Sus visuales no responden a los filtros de barrio; indícalo. Para métricas filtrables usa las medidas basadas en `fact_oferta`.\n\nNo relaciones dimensiones entre sí. Revisa las relaciones detectadas automáticamente y evita rutas adicionales o muchos-a-muchos. Oculta las claves técnicas. Ordena `dim_capacidad[grupo_capacidad]` por `orden` y `dim_fecha[mes_nombre]` por `mes`.\n\n## 3. Medidas y formatos\n\nAbre `medidas_airbnb.dax` y crea cada medida por separado con Modelado → Nueva medida. El archivo completo no se pega como una sola medida. Los nombres de tablas deben coincidir con el apartado 1.\n\nFormato moneda BRL/R$ para precios, MAE, RMSE y error mediano; 2 decimales. Formato porcentaje para cobertura, mejora MAE y desviación relativa. R² es un número decimal, no un porcentaje de aciertos.\n\nNo uses SUMA para precios ni para latitud/longitud. Los indicadores de error solo contienen datos para prueba; los registros de desarrollo y sin precio tienen errores NULL. No presentes diferencias de desarrollo como evidencia de precisión fuera de muestra.\n\n## 4. Páginas propuestas\n\n### Mercado y oferta\n\nTarjetas: Anuncios, Anfitriones, Anuncios con precio y Cobertura de precio. Barras: anuncios por barrio y por tipo de alojamiento. Segmentadores: barrio, tipo, propiedad y capacidad. Muestra el corte y las fechas de recopilación. Son anuncios observados; no equivalen a reservas ni a viviendas únicas verificadas.\n\n### Precios y segmentos\n\nTarjetas: Mediana y Promedio del precio publicado, Cobertura de precio. Matriz: barrio → tipo → grupo de capacidad, con anuncios, anuncios con precio y mediana. Incluye el tamaño de cada grupo; 30 precios es una regla de presentación, no una garantía estadística. Utiliza BRL. Las medianas de grupos no son aditivas.\n\nSi usas un mapa, utiliza latitud y longitud de la tabla de hechos, con categoría de datos correspondiente y sin sumar. Los contornos o nombres de barrio no verifican por sí solos la pertenencia espacial.\n\n### Comparación experimental de precios\n\nTabla de detalle: anuncio_id, barrio, tipo, capacidad, precio publicado, precio estimado, desviación relativa, partición del modelo y soporte bajo. Encabezado: **Estimación experimental del precio anunciado**. El signo de la desviación solo describe diferencia frente al modelo; no demuestra sobreprecio, infravaloración ni rentabilidad.\n\nIdentifica los casos de entrenamiento/validación, los de prueba y los que no tienen precio observado. No mezcles sus errores en una tarjeta. Un segmento con menos de 30 comparables de desarrollo se marca con soporte bajo, no con una probabilidad de error.\n\n### Evaluación y calidad\n\nTarjetas dinámicas de prueba: Anuncios evaluados, MAE, MAE referencia, Mejora MAE, Error mediano, RMSE y R². En otro bloque, claramente rotulado, muestra MAE de sensibilidad y su población. Añade alertas de capacidad, noches y precio ausente. Usa `importancia_variables` para una barra con el aumento del MAE al permutar; no es una interpretación causal.\n\nControles iniciales sin filtros: 48 247 anuncios, 44 147 con precio, 4 100 sin precio, 9 206 anuncios evaluados, MAE BRL 594,06, MAE referencia BRL 758,79, mejora 21,71 %, RMSE BRL 8 757,65 y R² 0,01514. Sensibilidad: 9 098 anuncios y MAE BRL 297,94. Estos valores deben coincidir con los informes de la ejecución recibida; si cambias el modelo, revisa la versión y los nuevos controles.\n\n## 5. Límites que deben quedar visibles\n\nEl precio es anunciado. El modelo fue evaluado en anfitriones reservados del mismo corte ya explorado, sin validación temporal o externa. Su R² global es bajo; las estimaciones acompañan el análisis descriptivo y requieren revisión individual. El subconjunto hasta P99 no sustituye la evaluación completa.\n\n`dim_fecha` contiene un calendario para relacionar la fecha de recopilación. No convierte este corte en una serie histórica ni permite afirmar cambios mensuales, estacionalidad o demanda. La disponibilidad del anuncio no es ocupación observada.\n\n## 6. Guardar y revisar\n\nGuarda el informe como `powerbi/Airbnb_Rio_Analisis.pbix`. Verifica primero tarjetas sin filtros y luego un filtro de barrio. Las medidas de prueba pueden quedar en blanco si un filtro no deja casos evaluados; no sustituyas esos blancos por un resultado inventado.\n\nDocumentación: https://learn.microsoft.com/en-us/power-query/connectors/postgresql y https://learn.microsoft.com/en-us/power-bi/guidance/star-schema\n'
MEDIDAS_DAX='// Crear cada medida por separado en Power BI. Los porcentajes devuelven fracciones.\nAnuncios = COUNTROWS(fact_oferta)\n\nAnfitriones = DISTINCTCOUNT(fact_oferta[anfitrion_key])\n\nAnuncios con precio = CALCULATE([Anuncios], fact_oferta[precio_publicado] > 0)\n\nCobertura de precio = DIVIDE([Anuncios con precio], [Anuncios])\n\nAnuncios sin precio = CALCULATE([Anuncios], fact_oferta[sin_precio] = TRUE())\n\nMediana precio publicado = CALCULATE(MEDIAN(fact_oferta[precio_publicado]), fact_oferta[precio_publicado] > 0)\n\nPromedio precio publicado = CALCULATE(AVERAGE(fact_oferta[precio_publicado]), fact_oferta[precio_publicado] > 0)\n\nMediana precio estimado = MEDIAN(fact_oferta[precio_estimado])\n\nAnuncios evaluados prueba = COUNT(fact_oferta[error_absoluto_prueba])\n\nMAE prueba BRL = AVERAGE(fact_oferta[error_absoluto_prueba])\n\nRMSE prueba BRL = SQRT(AVERAGE(fact_oferta[error_cuadratico_prueba]))\n\nError mediano prueba BRL = MEDIAN(fact_oferta[error_absoluto_prueba])\n\nMAE referencia prueba BRL = AVERAGE(fact_oferta[error_referencia_prueba])\n\nMejora MAE vs referencia = DIVIDE([MAE referencia prueba BRL] - [MAE prueba BRL], [MAE referencia prueba BRL])\n\nR2 prueba =\nVAR Casos = FILTER(fact_oferta, NOT ISBLANK(fact_oferta[error_absoluto_prueba]))\nVAR MediaReal = AVERAGEX(Casos, fact_oferta[precio_publicado])\nVAR SSE = SUMX(Casos, fact_oferta[error_cuadratico_prueba])\nVAR SST = SUMX(Casos, POWER(fact_oferta[precio_publicado] - MediaReal, 2))\nRETURN IF(COUNTROWS(Casos) > 1 && SST > 0, 1 - DIVIDE(SSE, SST), BLANK())\n\nMAE prueba sensibilidad BRL = CALCULATE([MAE prueba BRL], fact_oferta[prueba_sensibilidad] = TRUE())\n\nAnuncios prueba sensibilidad = CALCULATE([Anuncios evaluados prueba], fact_oferta[prueba_sensibilidad] = TRUE())\n\nAnuncios con alerta capacidad = CALCULATE([Anuncios], fact_oferta[alerta_capacidad] = TRUE())\n\nAnuncios con alerta noches = CALCULATE([Anuncios], fact_oferta[alerta_noches] = TRUE())\n\nAnuncios con soporte bajo = CALCULATE([Anuncios], fact_oferta[soporte_bajo_segmento] = TRUE())\n\nMediana desviacion al modelo = MEDIAN(fact_oferta[desviacion_relativa_modelo])\n'
(POWERBI/'GUIA_POWERBI.md').write_text(GUIA_POWERBI,encoding='utf-8')
(POWERBI/'medidas_airbnb.dax').write_text(MEDIDAS_DAX,encoding='utf-8')
print('Guía:',POWERBI/'GUIA_POWERBI.md')
print('Medidas:',POWERBI/'medidas_airbnb.dax')

Guía: C:\Users\user\Documents\Inteligencia de negocios\Proyecto_IN\Proy_Analysis_Airbnb_Business_Model\powerbi\GUIA_POWERBI.md
Medidas: C:\Users\user\Documents\Inteligencia de negocios\Proyecto_IN\Proy_Analysis_Airbnb_Business_Model\powerbi\medidas_airbnb.dax


## 10. Verificar desde PostgreSQL y cerrar conexiones

Después actualiza el árbol de pgAdmin para ver `Schemas → bi → Tables`. Comparte los tres informes de `reports/powerbi/` antes de continuar con la construcción visual del PBIX.

In [10]:
try:
    with motor.connect() as conexion:
        with conexion.begin():
            conexion.execute(text('SET TRANSACTION READ ONLY'))
            comprobacion=pd.read_sql_query(text("""
                SELECT COUNT(*) AS anuncios,
                       COUNT(*) FILTER (WHERE precio_publicado > 0) AS anuncios_con_precio,
                       COUNT(*) FILTER (WHERE precio_publicado IS NULL) AS anuncios_sin_precio,
                       COUNT(error_absoluto_prueba) AS anuncios_prueba,
                       AVG(error_absoluto_prueba) AS mae_prueba_brl,
                       AVG(error_referencia_prueba) AS mae_referencia_prueba_brl
                FROM bi.fact_oferta
            """),conexion)
    comprobacion.to_csv(REPORTES/'comprobacion_postgresql_bi.csv',index=False,encoding='utf-8-sig')
    display(comprobacion)
finally:
    motor.dispose()
print('Preparación terminada. Continúa con powerbi/GUIA_POWERBI.md.')

,anuncios,anuncios_con_precio,anuncios_sin_precio,anuncios_prueba,mae_prueba_brl,mae_referencia_prueba_brl
0,48247,44147,4100,9206,594.060335,758.794134


Preparación terminada. Continúa con powerbi/GUIA_POWERBI.md.


## Qué compartir

- `reports/powerbi/resumen_powerbi.json`
- `reports/powerbi/catalogo_tablas_bi.csv`
- `reports/powerbi/comprobacion_postgresql_bi.csv`

La siguiente etapa es crear el archivo `Airbnb_Rio_Analisis.pbix` con la guía y comprobar tarjetas, relaciones e interacción de filtros. La estimación se rotulará experimental y la evaluación completa permanecerá visible.

**Alcance de la validación del archivo entregado:** sintaxis, integración, dimensiones y métricas comprobadas con datos sintéticos. La escritura en tu PostgreSQL y la ejecución de DAX en Power BI se verificarán en tu equipo.